# Ce qui fait bouger les prix : l'écart au consensus, pas le niveau · *What moves prices: the gap to consensus, not the level*

Notebook compagnon du chapitre **4. Les grandes variables macro qui font bouger vos placements : un panorama** — [lire l'article](https://nmlab.io/ressources/les-grandes-variables-macro-qui-font-bouger-vos-placements).
Companion notebook to chapter **4. The Big Macro Variables That Move Your Investments: A Panorama** — [read the article](https://nmlab.io/en/ressources/the-big-macro-variables-that-move-your-investments).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure est régénérée par le code — un **schéma éditable** : changez les libellés à votre guise. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure from code — an **editable diagram**: change the labels as you like; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


def surprise_panels(lang: str) -> list[dict]:
    """Deux rapports imaginaires : structure (consensus, publié, couleur, sens) + textes localisés.
    Two imaginary reports: structure (consensus, published, color, direction) + localized text."""
    if lang == "fr":
        panels = [
            dict(title="Rapport A \u2014 un « bon » chiffre qui déçoit",
                 subtitle="250 000 emplois créés : solide dans l'absolu\u2026",
                 cons_label="consensus\n300 k", pub_label="publié\n250 k",
                 concl="sous les attentes \u2192 le prix BAISSE"),
            dict(title="Rapport B \u2014 un chiffre médiocre qui soulage",
                 subtitle="80 000 emplois créés : faible dans l'absolu\u2026",
                 cons_label="consensus\n30 k", pub_label="publié\n80 k",
                 concl="au-dessus des attentes \u2192 le prix MONTE"),
        ]
    else:
        panels = [
            dict(title="Report A \u2014 a « good » number that disappoints",
                 subtitle="250,000 jobs created: solid in absolute terms\u2026",
                 cons_label="consensus\n300k", pub_label="published\n250k",
                 concl="below expectations \u2192 the price FALLS"),
            dict(title="Report B \u2014 a mediocre number that relieves",
                 subtitle="80,000 jobs created: weak in absolute terms\u2026",
                 cons_label="consensus\n30k", pub_label="published\n80k",
                 concl="above expectations \u2192 the price RISES"),
        ]
    struct = [dict(consensus=300, published=250, color="rose", direction="down"),
              dict(consensus=30, published=80, color="blue", direction="up")]
    for p, s in zip(panels, struct):
        p.update(s)
    return panels


from matplotlib.figure import Figure

LABELS = {
    "fr": dict(title="Ce qui fait bouger les prix : l'écart au consensus, pas le niveau",
               sub="Deux rapports sur l'emploi imaginaires \u2014 créations d'emplois mensuelles, en milliers",
               note="Le niveau finit par compter \u2014 il façonne les bénéfices sur la durée ; mais la réaction du jour\n"
                    "se joue sur la surprise, car ce qui était attendu est déjà dans les prix."),
    "en": dict(title="What moves prices: the gap to consensus, not the level",
               sub="Two imaginary jobs reports \u2014 monthly job creation, in thousands",
               note="The level ends up mattering \u2014 it shapes profits over time; but the day's reaction turns on\n"
                    "the surprise, for what was expected is already in the price."),
}

def build_figure(panels: list[dict], lang: str) -> Figure:
    """Schéma : deux droites graduées où le prix réagit à l'écart publié-consensus, pas au niveau."""
    text = LABELS[lang]
    palette = {"blue": nm.COLORS["blue"], "rose": nm.COLORS["rose"]}
    fig = nm.figure(height_px=880)
    ax = nm.blank_axes(fig)
    regions = [(160, 820), (940, 1600)]
    vmax, line_y = 380.0, 405
    for (x0, x1), p in zip(regions, panels):
        cx = (x0 + x1) / 2
        color = palette[p["color"]]

        def X(v, x0=x0, x1=x1):
            return x0 + (v / vmax) * (x1 - x0)

        ax.text(cx, 640, p["title"], ha="center", va="center",
                fontsize=24, fontweight="bold", color=nm.COLORS["text"])
        ax.text(cx, 578, p["subtitle"], ha="center", va="center", fontsize=20, color=nm.COLORS["muted"])

        ax.plot([X(0), X(vmax)], [line_y, line_y], color=nm.COLORS["edge"], lw=2.6, solid_capstyle="round")
        for t in (0, 100, 200, 300):
            ax.plot([X(t), X(t)], [line_y - 10, line_y + 10], color=nm.COLORS["edge"], lw=2.0)
            ax.text(X(t), line_y - 40, str(t), ha="center", va="top", fontsize=19, color=nm.COLORS["muted"])

        ax.scatter([X(p["consensus"])], [line_y], s=280, facecolors=nm.COLORS["bg"],
                   edgecolors=nm.COLORS["text"], linewidths=2.6, zorder=4)
        ax.text(X(p["consensus"]), line_y + 66, p["cons_label"], ha="center", va="bottom",
                fontsize=20, color=nm.COLORS["text"], linespacing=1.3)
        ax.scatter([X(p["published"])], [line_y], s=280, color=color, zorder=5)
        ax.annotate("", xy=(X(p["consensus"]), line_y + 34), xytext=(X(p["published"]), line_y + 34),
                    arrowprops=dict(arrowstyle="<->", color=color, lw=2.2))
        ax.text(X(p["published"]), line_y - 70, p["pub_label"], ha="center", va="top",
                fontsize=20, fontweight="bold", color=color, linespacing=1.3)

        marker = "^" if p["direction"] == "up" else "v"
        ax.scatter([x1 - 50], [line_y - 95], s=560, marker=marker, color=color, zorder=5)
        ax.text(cx, 165, p["concl"], ha="center", va="center", fontsize=23, fontweight="bold", color=color)

    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(surprise_panels(LANG), LANG)